# Course 08: Production Machine Learning Systems

**Companion notebook for**: `08-production-ml-systems.html`

## Learning Objectives
- Detect training-serving skew by comparing feature distributions
- Train models with mixed precision using `tf.keras.mixed_precision`
- Optimize models for deployment: quantization and pruning with TF Model Optimization Toolkit
- Convert Keras models to TensorFlow Lite for edge deployment
- Build a batch prediction pipeline concept using pandas and model.predict
- Measure and compare inference latency across optimization levels

## Exam Relevance
- **Section 3**: Scaling prototypes into ML models
- **Section 4**: Serving and scaling models
- **Section 6**: ML solutions monitoring

## Prerequisites
- Python 3.10+
- TensorFlow 2.12+
- tensorflow-model-optimization

---
## Setup & Dependencies

In [ ]:
%pip install -q tensorflow tensorflow-model-optimization numpy matplotlib scipy

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import tempfile

print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {len(tf.config.list_physical_devices('GPU'))}")

---
## Section 1: Training-Serving Skew Detection

**Training-serving skew** occurs when data or feature transformations differ between training and serving. This section demonstrates how to detect skew by comparing feature distributions using statistical tests.

### Common Causes
- Duplicate preprocessing code (Python for training, Java for serving)
- Stale normalization statistics
- Different data sources for training vs serving

### Detection Methods
- **KS Test** (Kolmogorov-Smirnov): tests if two samples come from the same distribution
- **PSI** (Population Stability Index): measures distribution shift magnitude
- **Schema validation**: TFDV checks for feature type/range violations

In [ ]:
from scipy import stats


# --- Simulate training data vs serving data with skew ---
np.random.seed(42)

# Training data: features have known distributions
n_train = 10000
train_features = {
    "age":     np.random.normal(35, 10, n_train),
    "income":  np.random.lognormal(10.5, 0.8, n_train),
    "score":   np.random.uniform(300, 850, n_train),
}

# Serving data: some features have drifted
n_serve = 5000
serve_features = {
    "age":     np.random.normal(35, 10, n_serve),       # No drift
    "income":  np.random.lognormal(11.0, 0.9, n_serve), # Drifted! Mean shifted
    "score":   np.random.uniform(300, 850, n_serve),     # No drift
}

print("Feature Distribution Comparison (KS Test)")
print("=" * 60)
print(f"{'Feature':<12} {'KS Stat':>10} {'p-value':>12} {'Drift?':>10}")
print("-" * 60)

for feature in train_features:
    ks_stat, p_value = stats.ks_2samp(
        train_features[feature],
        serve_features[feature]
    )
    drift = "YES" if p_value < 0.05 else "No"
    print(f"{feature:<12} {ks_stat:>10.4f} {p_value:>12.6f} {drift:>10}")

print("\nKS test detects that 'income' distribution has shifted between")
print("training and serving data. This would cause training-serving skew.")

In [ ]:
# --- Population Stability Index (PSI) ---

def calculate_psi(expected, actual, bins=10):
    """Calculate Population Stability Index.
    
    PSI < 0.1: no significant shift
    PSI 0.1-0.2: moderate shift, investigate
    PSI > 0.2: significant shift, action required
    """
    # Create bins based on expected (training) distribution
    breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf
    
    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts = np.histogram(actual, bins=breakpoints)[0]
    
    # Convert to proportions (add small epsilon to avoid log(0))
    expected_pct = expected_counts / len(expected) + 1e-6
    actual_pct = actual_counts / len(actual) + 1e-6
    
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi


print("Population Stability Index (PSI)")
print("=" * 55)
print(f"{'Feature':<12} {'PSI':>8} {'Interpretation':>25}")
print("-" * 55)

for feature in train_features:
    psi = calculate_psi(train_features[feature], serve_features[feature])
    if psi < 0.1:
        interp = "No significant shift"
    elif psi < 0.2:
        interp = "Moderate shift"
    else:
        interp = "SIGNIFICANT shift"
    print(f"{feature:<12} {psi:>8.4f} {interp:>25}")

print("\nPSI thresholds: <0.1 OK | 0.1-0.2 investigate | >0.2 action needed")

In [ ]:
# --- Visualize the distribution shift ---

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, feature in enumerate(train_features):
    ax = axes[idx]
    ax.hist(train_features[feature], bins=50, alpha=0.6,
            label="Training", color="#3b82f6", density=True)
    ax.hist(serve_features[feature], bins=50, alpha=0.6,
            label="Serving", color="#ef4444", density=True)
    
    psi = calculate_psi(train_features[feature], serve_features[feature])
    ks_stat, p_val = stats.ks_2samp(train_features[feature], serve_features[feature])
    
    ax.set_title(f"{feature}\nPSI={psi:.4f}, KS p={p_val:.4f}", fontsize=11)
    ax.legend(fontsize=9)
    ax.set_ylabel("Density")
    
    if psi > 0.1:
        ax.set_facecolor("#fff5f5")

plt.suptitle("Training vs Serving Feature Distributions (Skew Detection)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("The 'income' feature shows visible distribution shift (red background).")
print("This type of skew silently degrades model predictions in production.")

---
## Section 2: Mixed Precision Training

**Mixed precision** uses float16 for most operations and float32 for numerically sensitive operations (loss computation, weight updates). Benefits:
- **2x memory reduction** (float16 = 2 bytes vs float32 = 4 bytes)
- **2-3x speedup** on GPUs with Tensor Cores (V100, A100, H100)
- **Minimal accuracy loss** when done correctly

### Key Rules
1. Set the global policy: `mixed_float16` (GPU) or `mixed_bfloat16` (TPU)
2. The **output layer must produce float32** (softmax in float16 causes numerical issues)
3. Loss scaling is handled automatically in TF 2.x

In [ ]:
# --- Load Fashion MNIST for comparison ---

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(f"Training: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# --- Build and train with float32 (baseline) ---

tf.keras.mixed_precision.set_global_policy("float32")

def build_cnn(name="model"):
    """Build a simple CNN for Fashion MNIST."""
    return tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, 3, activation="relu", input_shape=(28, 28, 1)),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        # Output layer: always float32 for numerical stability
        tf.keras.layers.Dense(10, activation="softmax", dtype="float32")
    ], name=name)


# Float32 training
model_fp32 = build_cnn("fp32_model")
model_fp32.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Training with float32 (baseline):")
start = time.perf_counter()
hist_fp32 = model_fp32.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    verbose=1
)
fp32_time = time.perf_counter() - start
fp32_acc = model_fp32.evaluate(X_test, y_test, verbose=0)[1]
print(f"\nFloat32 — Test accuracy: {fp32_acc:.4f}, Time: {fp32_time:.1f}s")

In [ ]:
# --- Build and train with mixed precision ---

# Set mixed precision policy
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print(f"Compute dtype: {tf.keras.mixed_precision.global_policy().compute_dtype}")
print(f"Variable dtype: {tf.keras.mixed_precision.global_policy().variable_dtype}")

model_mp = build_cnn("mixed_precision_model")
model_mp.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nTraining with mixed precision (float16 compute, float32 variables):")
start = time.perf_counter()
hist_mp = model_mp.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    verbose=1
)
mp_time = time.perf_counter() - start
mp_acc = model_mp.evaluate(X_test, y_test, verbose=0)[1]
print(f"\nMixed Precision — Test accuracy: {mp_acc:.4f}, Time: {mp_time:.1f}s")

# Reset policy
tf.keras.mixed_precision.set_global_policy("float32")

In [ ]:
# --- Compare results ---

print("Mixed Precision Comparison")
print("=" * 50)
print(f"{'Metric':<25} {'Float32':>12} {'Mixed FP16':>12}")
print("-" * 50)
print(f"{'Test Accuracy':<25} {fp32_acc:>12.4f} {mp_acc:>12.4f}")
print(f"{'Training Time (s)':<25} {fp32_time:>12.1f} {mp_time:>12.1f}")
print(f"{'Accuracy Difference':<25} {'--':>12} {abs(fp32_acc - mp_acc):>12.4f}")

# Note: speedup is minimal on CPU; on GPU with Tensor Cores expect 2-3x
print(f"\nNote: On CPU, mixed precision may not be faster (no Tensor Cores).")
print(f"On V100/A100 GPUs, expect 2-3x speedup with near-identical accuracy.")
print(f"\nExam tip: Output layer MUST be float32. Set dtype='float32' explicitly.")

---
## Section 3: Model Optimization — Quantization & Pruning

The **TensorFlow Model Optimization Toolkit** provides techniques to reduce model size and increase inference speed for deployment:

| Technique | Effect | Size Reduction | Speed Improvement |
|---|---|---|---|
| **Post-training quantization** | float32 → int8 weights | ~4x | Variable (CPU: 2-3x) |
| **Pruning** | Zero out low-magnitude weights | 2-6x (with compression) | Depends on hardware |
| **Quantization-aware training** | Train with simulated quantization | ~4x | 2-3x on CPU |

In [ ]:
# --- Train a baseline model for optimization ---

baseline_model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation="relu", input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

baseline_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

baseline_model.fit(X_train, y_train, epochs=5, batch_size=128,
                   validation_split=0.1, verbose=1)

baseline_acc = baseline_model.evaluate(X_test, y_test, verbose=0)[1]
print(f"\nBaseline accuracy: {baseline_acc:.4f}")

# Save baseline model
baseline_path = os.path.join(tempfile.gettempdir(), "baseline_model.keras")
baseline_model.save(baseline_path)
baseline_size = os.path.getsize(baseline_path)
print(f"Baseline model size: {baseline_size / 1024:.1f} KB")

In [ ]:
# --- Post-Training Quantization (dynamic range) ---

converter = tf.lite.TFLiteConverter.from_keras_model(baseline_model)

# Dynamic range quantization: float32 weights → int8
converter.optimizations = [tf.lite.Optimize.DEFAULT]

quantized_model = converter.convert()

# Save quantized model
quantized_path = os.path.join(tempfile.gettempdir(), "quantized_model.tflite")
with open(quantized_path, "wb") as f:
    f.write(quantized_model)

quantized_size = os.path.getsize(quantized_path)
print(f"Quantized model size: {quantized_size / 1024:.1f} KB")
print(f"Size reduction: {baseline_size / quantized_size:.1f}x")

In [ ]:
# --- Full Integer Quantization (int8 weights + activations) ---

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(baseline_model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]

# Representative dataset for calibration
def representative_dataset():
    """Provide sample inputs to calibrate quantization ranges."""
    for i in range(100):
        yield [X_train[i:i+1].astype(np.float32)]

converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.uint8
converter_int8.inference_output_type = tf.uint8

int8_model = converter_int8.convert()

int8_path = os.path.join(tempfile.gettempdir(), "int8_model.tflite")
with open(int8_path, "wb") as f:
    f.write(int8_model)

int8_size = os.path.getsize(int8_path)
print(f"Int8 model size: {int8_size / 1024:.1f} KB")
print(f"Size reduction vs baseline: {baseline_size / int8_size:.1f}x")

In [ ]:
# --- Pruning with TF Model Optimization Toolkit ---

import tensorflow_model_optimization as tfmot

# Apply pruning to the baseline model
pruning_params = {
    "pruning_schedule": tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.30,   # Start with 30% sparsity
        final_sparsity=0.80,     # End with 80% sparsity
        begin_step=0,
        end_step=2000
    )
}

pruned_model = tfmot.sparsity.keras.prune_low_magnitude(
    baseline_model, **pruning_params
)

pruned_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Pruning requires a callback to update masks during training
callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

print("Training with pruning (30% → 80% sparsity):")
pruned_model.fit(
    X_train, y_train,
    epochs=3,
    batch_size=128,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

pruned_acc = pruned_model.evaluate(X_test, y_test, verbose=0)[1]
print(f"\nPruned model accuracy: {pruned_acc:.4f}")
print(f"Accuracy difference from baseline: {abs(baseline_acc - pruned_acc):.4f}")

In [ ]:
# --- Strip pruning wrappers and convert to TFLite ---

stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

# Convert pruned model to TFLite with quantization
converter_pruned = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
converter_pruned.optimizations = [tf.lite.Optimize.DEFAULT]
pruned_tflite = converter_pruned.convert()

pruned_tflite_path = os.path.join(tempfile.gettempdir(), "pruned_quantized.tflite")
with open(pruned_tflite_path, "wb") as f:
    f.write(pruned_tflite)

pruned_tflite_size = os.path.getsize(pruned_tflite_path)

# --- Summary comparison ---
print("\nModel Optimization Summary")
print("=" * 65)
print(f"{'Model':<30} {'Size (KB)':>10} {'Reduction':>10} {'Accuracy':>10}")
print("-" * 65)
print(f"{'Baseline (Keras, FP32)':<30} {baseline_size/1024:>10.1f} {'1.0x':>10} {baseline_acc:>10.4f}")
print(f"{'Dynamic Range Quantized':<30} {quantized_size/1024:>10.1f} {f'{baseline_size/quantized_size:.1f}x':>10} {'~same':>10}")
print(f"{'Full Int8 Quantized':<30} {int8_size/1024:>10.1f} {f'{baseline_size/int8_size:.1f}x':>10} {'~same':>10}")
print(f"{'Pruned + Quantized':<30} {pruned_tflite_size/1024:>10.1f} {f'{baseline_size/pruned_tflite_size:.1f}x':>10} {pruned_acc:>10.4f}")
print("\nCombining pruning + quantization gives the best size reduction.")

---
## Section 4: TF Lite Conversion from Keras Model

TensorFlow Lite is optimized for on-device inference (mobile, IoT, edge). The conversion workflow:

1. Train a full Keras model
2. Convert to `.tflite` format using `TFLiteConverter`
3. Optionally quantize during conversion
4. Run inference using `tf.lite.Interpreter`

In [ ]:
# --- TF Lite conversion and inference ---

# Convert baseline Keras model to TF Lite (no quantization)
converter_base = tf.lite.TFLiteConverter.from_keras_model(baseline_model)
tflite_model = converter_base.convert()

tflite_path = os.path.join(tempfile.gettempdir(), "fashion_model.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"TFLite model saved: {os.path.getsize(tflite_path) / 1024:.1f} KB")

# --- Run inference with TF Lite Interpreter ---
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"\nInput shape:  {input_details[0]['shape']}")
print(f"Input dtype:  {input_details[0]['dtype']}")
print(f"Output shape: {output_details[0]['shape']}")

# Run inference on a single sample
sample = X_test[0:1].astype(np.float32)
interpreter.set_tensor(input_details[0]["index"], sample)
interpreter.invoke()
tflite_pred = interpreter.get_tensor(output_details[0]["index"])

# Compare with Keras prediction
keras_pred = baseline_model.predict(sample, verbose=0)

class_names = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print(f"\nKeras prediction:  {class_names[np.argmax(keras_pred)]} ({np.max(keras_pred):.4f})")
print(f"TFLite prediction: {class_names[np.argmax(tflite_pred)]} ({np.max(tflite_pred):.4f})")
print(f"Predictions match: {np.argmax(keras_pred) == np.argmax(tflite_pred)}")

In [ ]:
# --- Evaluate TFLite model accuracy on full test set ---

def evaluate_tflite(model_path, test_data, test_labels):
    """Evaluate a TFLite model on a test dataset."""
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    correct = 0
    total = len(test_labels)
    
    for i in range(total):
        sample = test_data[i:i+1].astype(np.float32)
        interpreter.set_tensor(input_details[0]["index"], sample)
        interpreter.invoke()
        pred = interpreter.get_tensor(output_details[0]["index"])
        if np.argmax(pred) == test_labels[i]:
            correct += 1
    
    return correct / total


# Evaluate on subset (full test set takes a while with single-sample inference)
subset_size = 1000
tflite_acc = evaluate_tflite(tflite_path, X_test[:subset_size], y_test[:subset_size])
print(f"TFLite accuracy on {subset_size} samples: {tflite_acc:.4f}")
print(f"Keras accuracy on same samples: {baseline_acc:.4f}")
print(f"\nAccuracy is preserved after TFLite conversion.")

---
## Section 5: Batch Prediction Pipeline Concept

Batch prediction is used when:
- Latency is not critical (nightly recommendations, periodic risk scoring)
- Input volume is large and bounded
- You want to precompute results for fast lookup at serving time

### Pattern
```
Cloud Scheduler → Vertex AI Pipeline → Read from BigQuery → model.predict() → Write to BigQuery
```

In [ ]:
import pandas as pd

# --- Simulate a batch prediction pipeline ---

# Step 1: Load batch data (simulating BigQuery read)
print("Step 1: Loading batch data (simulating BigQuery read)...")
n_items = 10000
np.random.seed(42)

# Simulate a dataset of items to score
batch_data = pd.DataFrame({
    "item_id": range(n_items),
    "pixel_data_idx": np.random.randint(0, len(X_test), n_items)
})
print(f"  Loaded {len(batch_data)} items to score.")

# Step 2: Predict in batches (memory-efficient)
print("Step 2: Running batch predictions...")
BATCH_SIZE = 256
all_predictions = []
all_confidences = []

start = time.perf_counter()
for i in range(0, len(batch_data), BATCH_SIZE):
    batch_indices = batch_data["pixel_data_idx"].iloc[i:i+BATCH_SIZE].values
    batch_images = X_test[batch_indices]
    
    preds = baseline_model.predict(batch_images, batch_size=BATCH_SIZE, verbose=0)
    predicted_classes = np.argmax(preds, axis=1)
    confidences = np.max(preds, axis=1)
    
    all_predictions.extend(predicted_classes)
    all_confidences.extend(confidences)

elapsed = time.perf_counter() - start
print(f"  Predicted {len(batch_data)} items in {elapsed:.2f}s ({len(batch_data)/elapsed:.0f} items/sec)")

# Step 3: Attach predictions to DataFrame
batch_data["predicted_class"] = all_predictions
batch_data["predicted_label"] = [class_names[c] for c in all_predictions]
batch_data["confidence"] = all_confidences

print("\nStep 3: Predictions attached to DataFrame (ready for BigQuery write)")
print(batch_data.head(10).to_string(index=False))

# Step 4: Summary statistics
print(f"\nPrediction Distribution:")
print(batch_data["predicted_label"].value_counts().head())
print(f"\nMean confidence: {batch_data['confidence'].mean():.4f}")
print(f"Low confidence (<0.7): {(batch_data['confidence'] < 0.7).sum()} items")

In [ ]:
# --- Vertex AI Batch Prediction (reference code) ---

batch_pred_code = '''
from google.cloud import aiplatform

aiplatform.init(project="my-project", location="us-central1")

# Get the deployed model
model = aiplatform.Model("projects/my-project/locations/us-central1/models/12345")

# Submit batch prediction job
batch_prediction_job = model.batch_predict(
    job_display_name="nightly-scoring",
    gcs_source="gs://my-bucket/input/items.jsonl",
    gcs_destination_prefix="gs://my-bucket/output/predictions/",
    machine_type="n1-standard-4",
    accelerator_type="NVIDIA_TESLA_T4",
    accelerator_count=1,
    starting_replica_count=2,
    max_replica_count=10,
)

batch_prediction_job.wait()
print(f"Output: {batch_prediction_job.output_info}")
'''

print("Vertex AI Batch Prediction — Reference Code")
print("=" * 50)
print(batch_pred_code)
print("NOTE: Running this incurs cloud costs.")

---
## Section 6: Latency Measurement & Optimization Patterns

Inference latency is critical for real-time serving. Key metrics:
- **P50 (median)**: typical latency
- **P95**: latency for 95% of requests (used in SLOs)
- **P99**: tail latency (worst-case for users)

Common optimizations:
1. **Model quantization** (float32 → int8)
2. **Batching** multiple requests together
3. **GPU acceleration** (TF Serving with GPU)
4. **Model pruning + distillation** for smaller models
5. **Caching** frequent predictions

In [ ]:
# --- Measure inference latency across optimization levels ---

def measure_latency(predict_fn, input_data, n_runs=100, warmup=10):
    """Measure inference latency with warmup runs."""
    # Warmup
    for _ in range(warmup):
        predict_fn(input_data)
    
    latencies = []
    for _ in range(n_runs):
        start = time.perf_counter()
        predict_fn(input_data)
        latencies.append((time.perf_counter() - start) * 1000)  # ms
    
    return np.array(latencies)


# Prepare single sample
single_sample = X_test[0:1].astype(np.float32)

# --- Keras model latency ---
keras_latencies = measure_latency(
    lambda x: baseline_model(x, training=False),
    tf.constant(single_sample)
)

# --- TFLite model latency ---
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_idx = interpreter.get_input_details()[0]["index"]
output_idx = interpreter.get_output_details()[0]["index"]

def tflite_predict(x):
    interpreter.set_tensor(input_idx, x)
    interpreter.invoke()
    return interpreter.get_tensor(output_idx)

tflite_latencies = measure_latency(tflite_predict, single_sample)

# --- Quantized TFLite model latency ---
interp_quant = tf.lite.Interpreter(model_path=quantized_path)
interp_quant.allocate_tensors()
q_input_idx = interp_quant.get_input_details()[0]["index"]
q_output_idx = interp_quant.get_output_details()[0]["index"]

def quantized_predict(x):
    interp_quant.set_tensor(q_input_idx, x)
    interp_quant.invoke()
    return interp_quant.get_tensor(q_output_idx)

quantized_latencies = measure_latency(quantized_predict, single_sample)

# --- Results ---
print("Inference Latency Comparison (single sample, CPU)")
print("=" * 70)
print(f"{'Model':<25} {'P50 (ms)':>10} {'P95 (ms)':>10} {'P99 (ms)':>10} {'Mean (ms)':>10}")
print("-" * 70)

for name, lats in [("Keras (FP32)", keras_latencies),
                    ("TFLite (FP32)", tflite_latencies),
                    ("TFLite (Quantized)", quantized_latencies)]:
    print(f"{name:<25} {np.percentile(lats, 50):>10.2f} "
          f"{np.percentile(lats, 95):>10.2f} "
          f"{np.percentile(lats, 99):>10.2f} "
          f"{np.mean(lats):>10.2f}")

In [ ]:
# --- Visualize latency distributions ---

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

data = [
    ("Keras (FP32)", keras_latencies, "#3b82f6"),
    ("TFLite (FP32)", tflite_latencies, "#22c55e"),
    ("TFLite (Quantized)", quantized_latencies, "#f97316"),
]

for (name, lats, color), ax in zip(data, axes):
    ax.hist(lats, bins=30, color=color, alpha=0.7, edgecolor="white")
    ax.axvline(np.percentile(lats, 50), color="red", linestyle="--",
               label=f"P50: {np.percentile(lats, 50):.2f}ms")
    ax.axvline(np.percentile(lats, 99), color="darkred", linestyle=":",
               label=f"P99: {np.percentile(lats, 99):.2f}ms")
    ax.set_title(name, fontsize=12, fontweight="bold")
    ax.set_xlabel("Latency (ms)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)

plt.suptitle("Inference Latency Distribution by Optimization Level",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Key insight: TFLite conversion alone often provides significant speedup.")
print("Quantization adds further improvement, especially on CPU (no Tensor Cores).")
print("\nExam tip: For latency-sensitive serving, quantize models and use GPU endpoints.")

In [ ]:
# --- Batching effect on throughput ---

batch_sizes = [1, 4, 16, 32, 64, 128, 256]
throughputs = []

for bs in batch_sizes:
    batch = X_test[:bs].astype(np.float32)
    
    # Warmup
    for _ in range(5):
        baseline_model.predict(batch, batch_size=bs, verbose=0)
    
    # Measure
    start = time.perf_counter()
    for _ in range(20):
        baseline_model.predict(batch, batch_size=bs, verbose=0)
    elapsed = time.perf_counter() - start
    
    items_per_sec = (bs * 20) / elapsed
    throughputs.append(items_per_sec)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(batch_sizes, throughputs, "o-", color="#3b82f6", linewidth=2, markersize=8)
ax.set_xlabel("Batch Size", fontsize=12)
ax.set_ylabel("Throughput (items/sec)", fontsize=12)
ax.set_title("Throughput vs Batch Size (Keras, CPU)", fontsize=14, fontweight="bold")
ax.set_xscale("log", base=2)
ax.grid(True, alpha=0.3)

for bs, tp in zip(batch_sizes, throughputs):
    ax.annotate(f"{tp:.0f}", (bs, tp), textcoords="offset points",
               xytext=(0, 12), ha="center", fontsize=9)

plt.tight_layout()
plt.show()

print("Larger batch sizes improve throughput (better hardware utilization).")
print("But larger batches increase latency per individual prediction.")
print("\nTrade-off: Batch prediction = high throughput. Online = low latency.")

---
## Summary

In this notebook we covered:

1. **Training-Serving Skew Detection** — KS test and PSI for comparing training vs serving feature distributions. Use TFDV or Vertex AI Model Monitoring in production.

2. **Mixed Precision Training** — `mixed_float16` policy for 2x memory reduction and 2-3x GPU speedup. Output layer must be float32.

3. **Model Optimization** — Post-training quantization (float32 to int8, ~4x size reduction), pruning (80% sparsity with minimal accuracy loss), and combined techniques.

4. **TF Lite Conversion** — Convert Keras models to `.tflite` for edge deployment. Verify predictions match between Keras and TFLite.

5. **Batch Prediction Pipeline** — Pandas + model.predict pattern for offline scoring. Vertex AI Batch Prediction for managed infrastructure.

6. **Latency Optimization** — Measured P50/P95/P99 across optimization levels. Batching improves throughput but increases per-request latency.

### Exam Key Points
- **Training-serving skew**: prevent with tf.Transform or Keras preprocessing layers
- **Data drift vs concept drift**: distribution shift vs relationship change
- **Mixed precision**: output layer must be float32
- **TPU**: bfloat16, batch size multiple of 8, no Python-side preprocessing
- **Batch prediction**: non-latency-critical, large volume. **Online**: low latency, user-facing

**Next notebook**: Course 09 covers MLOps: Getting Started.